# Complete Flash Attention Model with REE Training Pipeline

Self-contained end-to-end training pipeline for Flash Attention-based transformer model with:
- **Rotary Expression Embeddings (REE)**: Expression values modulate rotary positional encodings
- **Gene Token IDs**: Unique token per ortholog gene (no CLS token)
- **Multi-Head Self-Attention**: Scaled dot-product attention with 8 heads
- **Max Pooling**: Sample-level representation aggregation
- **Training & Validation**: Full DDP-ready training loop
- **Evaluation**: Embeddings, metrics, and visualization

**Pipeline Overview:**
1. Load orthologs and create tokenization scheme
2. Load and preprocess ARCHS4 expression data (TPM normalization)
3. Build tokenized sequences with expression values
4. Create train/val/test splits
5. Define Flash Attention model with REE
6. Training loop with multi-head attention
7. Evaluation and embedding generation
8. Save model checkpoints and artifacts

In [3]:
"""
SECTION 1: Import Required Libraries and Configuration
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION & HYPERPARAMETERS
# ============================================================
print("=" * 70)
print("SECTION 1: CONFIGURATION & SETUP")
print("=" * 70)

# File paths
ORTHOLOGS_FILE = "data/ensembl/orthologs_one2one.txt"
ARCHS4_DIR = "data/archs4"
OUTPUT_DIR = "./models"
DATA_DIR = "./prepared_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Gene tokenization
GENE_ID_START = 1

# Model hyperparameters
EMBEDDING_DIM = 256
NUM_LAYERS = 4
NUM_HEADS = 8
FF_DIM = 1024
DROPOUT = 0.1
ROPE_BASE = 10000.0

# Training parameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Device: {DEVICE}")
print(f"✅ Configuration:")
print(f"   Embedding dim: {EMBEDDING_DIM}")
print(f"   Layers: {NUM_LAYERS}, Heads: {NUM_HEADS}")
print(f"   Batch size: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"   Epochs: {NUM_EPOCHS}")

SECTION 1: CONFIGURATION & SETUP

🖥️  Device: cuda
✅ Configuration:
   Embedding dim: 256
   Layers: 4, Heads: 8
   Batch size: 32, LR: 0.0001
   Epochs: 20


In [4]:
"""
SECTION 2: Load Orthologs and Create Gene Tokenization Scheme
"""

print("\n" + "=" * 70)
print("SECTION 2: ORTHOLOG LOADING & TOKENIZATION SCHEME")
print("=" * 70)

# Load orthologs
print(f"\n📂 Loading orthologs from {ORTHOLOGS_FILE}")
ortho_df = pd.read_csv(ORTHOLOGS_FILE, sep="\t")

# Extract human genes
human_ortho_genes = ortho_df["Human gene name"].unique().tolist()
mouse_ortho_genes = ortho_df["Gene name"].unique().tolist()

print(f"✅ Loaded {len(ortho_df):,} ortholog pairs")
print(f"   Human genes (total): {len(human_ortho_genes):,}")
print(f"   Mouse genes: {len(mouse_ortho_genes):,}")

# Filter out NaN values (missing human gene names)
human_ortho_genes_valid = [g for g in human_ortho_genes if isinstance(g, str)]
print(f"   Human genes (valid/named): {len(human_ortho_genes_valid):,}")

# Create bidirectional mapping
ortho_map = dict(zip(ortho_df["Gene name"], ortho_df["Human gene name"]))
human_to_mouse = dict(zip(ortho_df["Human gene name"], ortho_df["Gene name"]))

# Create tokenization scheme
print(f"\n🔤 Creating gene tokenization scheme...")
canonical_ortho_genes = sorted(human_ortho_genes_valid)
NUM_GENES_ORTHO = len(canonical_ortho_genes)

gene_to_token_id = {gene: GENE_ID_START + idx for idx, gene in enumerate(canonical_ortho_genes)}
token_id_to_gene = {token_id: gene for gene, token_id in gene_to_token_id.items()}

print(f"✅ Created tokenization scheme:")
print(f"   Total genes: {NUM_GENES_ORTHO:,}")
print(f"   Gene ID range: {GENE_ID_START} - {max(gene_to_token_id.values())}")
print(f"   Example mapping: {list(gene_to_token_id.items())[:5]}")

# Save canonical ortholog gene list to reference file
ensembl_dir = "../data/ensembl"
os.makedirs(ensembl_dir, exist_ok=True)
canonical_genes_path = os.path.join(ensembl_dir, "canonical_ortholog_genes.csv")
canonical_df = pd.DataFrame({
    "token_id": list(token_id_to_gene.keys()),
    "gene_symbol": list(token_id_to_gene.values())
})
canonical_df.to_csv(canonical_genes_path, index=False)
print(f"\n💾 Saved canonical ortholog genes → {canonical_genes_path}")

# Save tokenization mapping
token_mapping_df = pd.DataFrame({
    "token_id": list(token_id_to_gene.keys()),
    "gene_symbol": list(token_id_to_gene.values())
})
token_mapping_df.to_csv(os.path.join(DATA_DIR, "token_id_mapping.csv"), index=False)
print(f"💾 Saved tokenization mapping → {DATA_DIR}/token_id_mapping.csv")


SECTION 2: ORTHOLOG LOADING & TOKENIZATION SCHEME

📂 Loading orthologs from data/ensembl/orthologs_one2one.txt
✅ Loaded 16,168 ortholog pairs
   Human genes (total): 16,110
   Mouse genes: 16,156
   Human genes (valid/named): 16,109

🔤 Creating gene tokenization scheme...
✅ Created tokenization scheme:
   Total genes: 16,109
   Gene ID range: 1 - 16109
   Example mapping: [('A1CF', 1), ('A2M', 2), ('A3GALT2', 3), ('A4GALT', 4), ('A4GNT', 5)]

💾 Saved canonical ortholog genes → ../data/ensembl/canonical_ortholog_genes.csv
💾 Saved tokenization mapping → ./prepared_data/token_id_mapping.csv


In [7]:
"""
SECTION 3: Compute Canonical Exon Lengths from GENCODE v49 (Human & Mouse)
"""

print("\n" + "=" * 70)
print("SECTION 3: COMPUTE EXON LENGTHS FROM GENCODE v49")
print("=" * 70)

import pandas as pd
import pyranges as pr

# ============================================================
# COMPUTE EXON LENGTHS FOR HUMAN
# ============================================================
human_output = "data/gencode/gencode_v49_gene_exon_lengths.csv"

if os.path.exists(human_output):
    print(f"\n✅ Human exon lengths file already exists, loading...")
    exon_len_df_human = pd.read_csv(human_output)
    print(f"   ✅ Loaded exon lengths for {len(exon_len_df_human):,} HUMAN genes from {human_output}")
else:
    print(f"\n🧬 Computing exon lengths for HUMAN (GENCODE v49)...")

    gtf_path_human = "data/gencode/gencode.v49.basic.annotation.gtf.gz"

    # Load GTF
    gtf_human = pr.read_gtf(gtf_path_human)
    df_human = gtf_human.as_df()
    # print(gtf_human.columns)

    # Keep only exons
    exons_human = gtf_human[gtf_human.Feature == "exon"]

    # Convert to dataframe
    df_human = exons_human.as_df()[["Chromosome", "Start", "End", "gene_name"]]

    exon_lengths_human = {}

    for gene, subdf in df_human.groupby("gene_name"):
        gr = pr.PyRanges(subdf)
        merged = gr.merge()
        
        # merged.length may be int OR Series, handle both
        length = merged.length
        if hasattr(length, "sum"):
            total_len = int(length.sum())
        else:
            total_len = int(length)

        exon_lengths_human[gene] = total_len

    # Convert to DataFrame
    exon_len_df_human = pd.DataFrame(
        list(exon_lengths_human.items()),
        columns=["gene_symbol", "exon_length"]
    )

    exon_len_df_human.to_csv(human_output, index=False)

    print(f"   ✅ Saved exon lengths for {len(exon_len_df_human):,} HUMAN genes → {human_output}")

# ============================================================
# COMPUTE EXON LENGTHS FOR MOUSE (GRCm39 in v49)
# ============================================================
# Note: GENCODE vM38 is the standalone mouse GTF for GRCm39 coordinate system
# All genes in vM38 are already mouse genes, no filtering needed

mouse_output = "data/gencode/gencode_v49_mouse_gene_exon_lengths.csv"

if os.path.exists(mouse_output):
    print(f"\n✅ Mouse exon lengths file already exists, loading...")
    exon_len_df_mouse = pd.read_csv(mouse_output)
    print(f"   ✅ Loaded exon lengths for {len(exon_len_df_mouse):,} MOUSE genes from {mouse_output}")
else:
    print(f"\n🧬 Computing exon lengths for MOUSE from GENCODE vM38...")

    gtf_path_mouse = "data/gencode/gencode.vM38.basic.annotation.gtf.gz"

    # Load GTF
    gtf_mouse = pr.read_gtf(gtf_path_mouse)

    # Keep only exons
    exons_mouse = gtf_mouse[gtf_mouse.Feature == "exon"]

    # Convert to dataframe - get chromosome, coordinates, and gene name
    df_mouse = exons_mouse.as_df()[["Chromosome", "Start", "End", "gene_name"]]

    print(f"   Total exon records: {len(df_mouse):,}")

    # Since vM38 is standalone mouse GTF, all genes are already mouse
    # No need to filter by ENSMUST prefix
    exon_lengths_mouse = {}

    for gene, subdf in df_mouse.groupby("gene_name"):
        gr = pr.PyRanges(subdf)
        merged = gr.merge()
        
        # merged.length may be int OR Series, handle both
        length = merged.length
        if hasattr(length, "sum"):
            total_len = int(length.sum())
        else:
            total_len = int(length)

        exon_lengths_mouse[gene] = total_len

    # Convert to DataFrame
    exon_len_df_mouse = pd.DataFrame(
        list(exon_lengths_mouse.items()),
        columns=["gene_symbol", "exon_length"]
    )

    exon_len_df_mouse.to_csv(mouse_output, index=False)

    print(f"   ✅ Processed {len(exon_lengths_mouse):,} MOUSE genes from vM38")
    print(f"   ✅ Saved exon lengths → {mouse_output}")

print(f"\n✅ Exon length computation complete!")
print(f"   Human: {len(exon_len_df_human):,} genes")
print(f"   Mouse: {len(exon_len_df_mouse):,} genes")


SECTION 3: COMPUTE EXON LENGTHS FROM GENCODE v49

🧬 Computing exon lengths for HUMAN (GENCODE v49)...
   ✅ Saved exon lengths for 77,078 HUMAN genes → data/gencode/gencode_v49_gene_exon_lengths.csv

🧬 Computing exon lengths for MOUSE from GENCODE vM38...
   Total exon records: 673,401
   ✅ Processed 77,969 MOUSE genes from vM38
   ✅ Saved exon lengths → data/gencode/gencode_v49_mouse_gene_exon_lengths.csv

✅ Exon length computation complete!
   Human: 77,078 genes
   Mouse: 77,969 genes


In [9]:
"""
DIAGNOSTIC: Validate ARCHS4 HDF5 File Integrity
Checks for corruption and filter compatibility issues
"""

print("\n" + "=" * 70)
print("DIAGNOSTIC: VALIDATE ARCHS4 HDF5 FILES")
print("=" * 70)

import h5py
import numpy as np
import os

ARCHS4_DIR = "data/archs4"
human_h5_path = os.path.join(ARCHS4_DIR, "human_gene_v2.5.h5")
mouse_h5_path = os.path.join(ARCHS4_DIR, "mouse_gene_v2.5.h5")

def validate_h5file(h5_path, name):
    """Validate HDF5 file integrity"""
    print(f"\n{'='*70}")
    print(f"Testing {name}:")
    print(f"File: {h5_path}")
    print(f"Exists: {os.path.exists(h5_path)}")
    
    if not os.path.exists(h5_path):
        print(f"❌ FILE NOT FOUND")
        return False
    
    try:
        # Check if it's a valid HDF5
        is_h5 = h5py.is_hdf5(h5_path)
        print(f"Valid HDF5: {is_h5}")
        
        if not is_h5:
            print(f"❌ Not a valid HDF5 file")
            return False
        
        # Try to open and read metadata
        with h5py.File(h5_path, 'r') as f:
            print(f"✅ File opens successfully")
            print(f"  Root keys: {list(f.keys())}")
            print(f"  Meta keys: {list(f['meta'].keys())}")
            
            # Test reading gene symbols (small read)
            try:
                genes = f['meta']['genes']['symbol']
                print(f"  Gene symbols shape: {genes.shape}")
                first_5_genes = [x.decode('UTF-8') for x in genes[:5]]
                print(f"  First 5 genes: {first_5_genes}")
                print(f"✅ Gene metadata readable")
            except Exception as e:
                print(f"❌ Gene metadata read failed: {e}")
                return False
            
            # Test reading sample IDs
            try:
                samples = f['meta']['samples']['geo_accession']
                print(f"  Sample IDs shape: {samples.shape}")
                first_5_samples = [x.decode('UTF-8') for x in samples[:5]]
                print(f"  First 5 samples: {first_5_samples}")
                print(f"✅ Sample metadata readable")
            except Exception as e:
                print(f"❌ Sample metadata read failed: {e}")
                return False
            
            # Test expression data shape
            try:
                expr_shape = f['data']['expression'].shape
                print(f"  Expression shape: {expr_shape}")
                print(f"✅ Expression dataset accessible")
            except Exception as e:
                print(f"❌ Expression dataset failed: {e}")
                return False
            
            # Try reading a small slice
            try:
                small_read = np.array(f['data']['expression'][:100, :10])  # 100 genes, 10 samples
                print(f"  Small read (100 genes × 10 samples) successful")
                print(f"  Sample values (first 5): {small_read[:5, 0]}")
                print(f"✅ Small data read works")
            except Exception as e:
                print(f"⚠️  Small read failed: {e}")
                print(f"  This suggests compression filter issues or corruption")
                return False
        
        print(f"\n✅ {name} is VALID")
        return True
        
    except Exception as e:
        print(f"\n❌ {name} has issues: {e}")
        import traceback
        traceback.print_exc()
        return False

# Validate both files
human_valid = validate_h5file(human_h5_path, "HUMAN ARCHS4")
mouse_valid = validate_h5file(mouse_h5_path, "MOUSE ARCHS4")

print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Human valid: {human_valid}")
print(f"Mouse valid: {mouse_valid}")

if not (human_valid and mouse_valid):
    print(f"\n⚠️  HDF5 VALIDATION FAILED")
    print(f"You need to re-download fresh ARCHS4 files:")
    print(f"""
# Download fresh ARCHS4 files:
cd ../data/archs4/
wget https://s3.dev.maayanlab.cloud/releases/archs4/human_gene_v2.5.h5
wget https://s3.dev.maayanlab.cloud/releases/archs4/mouse_gene_v2.5.h5
    """)
else:
    print(f"\n✅ Both files are valid. Ready to proceed with Section 4.0 & 4.1")



DIAGNOSTIC: VALIDATE ARCHS4 HDF5 FILES

Testing HUMAN ARCHS4:
File: data/archs4/human_gene_v2.5.h5
Exists: True
Valid HDF5: True
✅ File opens successfully
  Root keys: ['data', 'meta']
  Meta keys: ['genes', 'info', 'samples']
  Gene symbols shape: (67186,)
  First 5 genes: ['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112']
✅ Gene metadata readable
  Sample IDs shape: (888821,)
  First 5 samples: ['GSM1000981', 'GSM1000982', 'GSM1000983', 'GSM1000984', 'GSM1000985']
✅ Sample metadata readable
  Expression shape: (67186, 888821)
✅ Expression dataset accessible
  Small read (100 genes × 10 samples) successful
  Sample values (first 5): [ 105    0 3028  705 2426]
✅ Small data read works

✅ HUMAN ARCHS4 is VALID

Testing MOUSE ARCHS4:
File: data/archs4/mouse_gene_v2.5.h5
Exists: True
Valid HDF5: True
✅ File opens successfully
  Root keys: ['data', 'meta']
  Meta keys: ['genes', 'info', 'samples']
  Gene symbols shape: (53511,)
  First 5 genes: ['Gnai3', 'Pbsn', 'Cdc45', 'H19', 'Scml2']
✅ Gen

In [10]:
"""
SECTION 4: Setup archs4py for Random Sample Extraction
Prepare archs4py library for efficient random extraction (no full metadata scan needed)
"""

print("\n" + "=" * 70)
print("SECTION 4: SETUP ARCHS4PY")
print("=" * 70)

import os
import numpy as np
import pandas as pd
import time

# Import archs4py
try:
    import archs4py as a4
    print("✅ archs4py imported successfully")
except ImportError:
    print("📦 Installing archs4py...")
    import subprocess
    subprocess.check_call(["pip", "install", "archs4py"])
    import archs4py as a4
    print("✅ archs4py installed and imported")

ARCHS4_DIR = "data/archs4"
human_h5_path = os.path.join(ARCHS4_DIR, "human_gene_v2.5.h5")
mouse_h5_path = os.path.join(ARCHS4_DIR, "mouse_gene_v2.5.h5")

print(f"\n✅ Setup complete. Ready for random sampling in Section 4.1")
print(f"   Will extract random samples + metadata on-the-fly (no full scan)")


SECTION 4: SETUP ARCHS4PY
✅ archs4py imported successfully

✅ Setup complete. Ready for random sampling in Section 4.1
   Will extract random samples + metadata on-the-fly (no full scan)


In [ ]:
"""
SECTION 4.1: Extract Random Samples with Vectorized Batch Processing
Fast streaming pipeline: extract → batch normalize → accumulate → save
(Based on proven fast pipeline from previous project)
WITH CROSS-SPLIT DEDUPLICATION to prevent data leakage
"""

print("\n" + "=" * 70)
print("SECTION 4.1: EXTRACT & NORMALIZE WITH VECTORIZED BATCHING")
print("=" * 70)

import gc
from pathlib import Path

# ============================================================
# 0. CONFIGURATION
# ============================================================
TRAIN_SAMPLES = 100_000
VAL_SAMPLES = 10_000
TEST_SAMPLES = 10_000
EXTRACTION_BATCH_SIZE = 10_000  # Larger batches for vectorized processing
QC_MIN_NONZERO = 14_000

print(f"\n⚙️ Configuration:")
print(f"   Train samples: {TRAIN_SAMPLES}")
print(f"   Val samples: {VAL_SAMPLES}")
print(f"   Test samples: {TEST_SAMPLES}")
print(f"   Batch size: {EXTRACTION_BATCH_SIZE} (vectorized processing)")
print(f"   QC threshold: {QC_MIN_NONZERO:,} non-zero genes")

# ============================================================
# 1. LOAD EXON LENGTHS
# ============================================================
print(f"\n📂 Loading exon lengths...")
exon_file_human = "data/gencode/gencode_v49_gene_exon_lengths.csv"
exon_file_mouse = "data/gencode/gencode_v49_mouse_gene_exon_lengths.csv"
exon_df_human = pd.read_csv(exon_file_human)
exon_df_mouse = pd.read_csv(exon_file_mouse)
gene_lengths_human = exon_df_human.set_index("gene_symbol")["exon_length"]
gene_lengths_mouse = exon_df_mouse.set_index("gene_symbol")["exon_length"]

print(f"   Human: {len(gene_lengths_human):,} genes")
print(f"   Mouse: {len(gene_lengths_mouse):,} genes")

# Canonical genes
canonical_ortho_genes_list = canonical_ortho_genes  # From section 2

# ============================================================
# 2. OUTPUT DIRECTORY
# ============================================================
output_dir = os.path.join(ARCHS4_DIR, "train_orthologs")
os.makedirs(output_dir, exist_ok=True)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

print(f"\n✅ Output directory: {output_dir}")
print(f"🧬 Canonical genes: {len(canonical_ortho_genes_list):,}")

# ============================================================
# 3. VECTORIZED BATCH PROCESSING FUNCTION WITH CROSS-SPLIT DEDUPLICATION
# ============================================================
def extract_and_normalize_streaming(
    h5_file,
    species_name,
    split_name,
    n_samples,
    gene_lengths,
    canonical_genes,
    output_parquet,
    output_metadata,
    global_seen_samples,
    batch_size=10000,
    random_seed=42
):
    """
    Stream random samples in batches, process vectorized, accumulate, save once.
    Includes deduplication:
    - Within batches (local duplicates)
    - Across splits (global seen_samples from train/val/test)
    
    Args:
        global_seen_samples: Set of all samples already extracted in previous splits
        Returns tuple: (count, metadata_df, updated_seen_samples_set)
    """
    
    print(f"\n{'='*70}")
    print(f"🔄 {species_name} - {split_name.upper()}")
    print(f"{'='*70}")
    
    # Determine total samples
    if isinstance(n_samples, str) and n_samples.lower() == "all":
        n_extract = int(1e10)
        print(f"Extracting ALL available samples (vectorized batches of {batch_size:,})")
    else:
        n_extract = n_samples
        print(f"Extracting {n_extract:,} samples (vectorized batches of {batch_size:,})")
    
    if global_seen_samples:
        print(f"⚠️  Cross-split deduplication ACTIVE ({len(global_seen_samples):,} samples already in other splits)")
    
    batches = []  # Accumulate all batches in memory
    seen_samples_local = set()  # Track samples extracted in THIS pass (for within-batch dedup)
    total_processed = 0
    batch_num = 0
    
    while total_processed < n_extract:
        batch_num += 1
        samples_to_extract = min(batch_size, n_extract - total_processed)
        
        print(f"\n  📦 Batch {batch_num}: Requesting {samples_to_extract:,} samples...")
        
        try:
            # Extract batch
            expr_batch = a4.data.rand(
                h5_file,
                samples_to_extract,
                remove_sc=True,
                seed=random_seed + batch_num
            )
            
            if expr_batch.empty or expr_batch.shape[1] == 0:
                print(f"     ⚠️  No more samples. Stopping.")
                break
            
            print(f"     ✅ Received {expr_batch.shape[1]:,} samples")
            
            # Aggregate duplicate genes (vectorized)
            expr_batch = expr_batch.groupby(level=0).sum()
            print(f"     ✅ Aggregated gene duplicates: {expr_batch.shape[0]:,} unique genes")
            
        except Exception as e:
            print(f"     ❌ Error: {e}")
            break
        
        # ====== VECTORIZED BATCH PROCESSING ======
        # Keep only genes with exon lengths
        expr_batch = expr_batch.loc[expr_batch.index.intersection(gene_lengths.index)]
        
        # QC: Filter samples with too many zeros (vectorized)
        nonzero = (expr_batch > 0).sum(axis=0)
        valid_samples = nonzero[nonzero >= QC_MIN_NONZERO].index.tolist()
        expr_batch = expr_batch[valid_samples]
        failed_qc = expr_batch.shape[1] - len(valid_samples)
        
        if len(valid_samples) == 0:
            print(f"     ⚠️  All samples failed QC.")
            continue
        
        # ====== DEDUPLICATION (LOCAL): Remove samples already extracted in this split ======
        new_samples_local = [s for s in expr_batch.columns if s not in seen_samples_local]
        expr_batch = expr_batch[new_samples_local]
        duplicates_local = len(valid_samples) - len(new_samples_local)
        
        if len(new_samples_local) == 0:
            print(f"     ⚠️  All {len(valid_samples):,} samples were already extracted in this split. Skipping batch.")
            continue
        
        # ====== DEDUPLICATION (GLOBAL): Remove samples already in other splits ======
        new_samples_global = [s for s in expr_batch.columns if s not in global_seen_samples]
        duplicates_global = len(new_samples_local) - len(new_samples_global)
        expr_batch = expr_batch[new_samples_global]
        
        if len(new_samples_global) == 0:
            print(f"     ⚠️  All {len(new_samples_local):,} samples already in other splits! Skipping batch.")
            continue
        
        # Print summary with all dedup stats
        print(f"     ✅ After QC & dedup: {len(new_samples_global):,} new")
        if failed_qc > 0:
            print(f"        (failed_qc={failed_qc:,}, local_dups={duplicates_local:,}, global_dups={duplicates_global:,})")
        elif duplicates_local > 0 or duplicates_global > 0:
            print(f"        (local_dups={duplicates_local:,}, global_dups={duplicates_global:,})")
        
        # TPM normalization (vectorized)
        lengths_kb = (gene_lengths.loc[expr_batch.index].fillna(1000)) / 1000.0
        rate = expr_batch.div(lengths_kb, axis=0)
        expr_tpm = rate.div(rate.sum(axis=0), axis=1) * 1e6
        
        # Log transform (vectorized)
        expr_log = np.log1p(expr_tpm)
        
        # Align to canonical genes (vectorized)
        expr_aligned = expr_log.reindex(canonical_genes, fill_value=0).astype("float32")
        
        batches.append(expr_aligned)
        seen_samples_local.update(expr_aligned.columns)  # Track for local dedup
        total_processed += len(new_samples_global)
        
        # Clean up
        del expr_batch, expr_tpm, expr_log, expr_aligned
        gc.collect()
        
        print(f"     📊 Progress: {total_processed:,} total new samples extracted")
    
    # ====== COMBINE ALL BATCHES & SAVE ONCE ======
    if not batches:
        print(f"\n❌ No valid batches for {species_name}!")
        return 0, None, global_seen_samples
    
    print(f"\n💾 Combining {len(batches):,} batches and saving...")
    try:
        expr_combined = pd.concat(batches, axis=1)
    except ValueError as e:
        # Additional safety check for duplicate columns
        if "Duplicate column names" in str(e):
            print(f"     ⚠️  WARNING: Duplicate columns found despite deduplication!")
            print(f"     Attempting to remove duplicates before save...")
            # Deduplicate by keeping first occurrence of each sample
            expr_combined = pd.concat(batches, axis=1)
            expr_combined = expr_combined.loc[:, ~expr_combined.columns.duplicated(keep='first')]
            print(f"     ✅ Removed duplicates. Final shape: {expr_combined.shape}")
        else:
            raise
    
    # Save expression
    expr_combined.to_parquet(output_parquet, compression="zstd")
    print(f"   ✅ Saved expression: {expr_combined.shape} → {output_parquet}")
    
    # Create and save metadata
    metadata_df = pd.DataFrame({
        "geo_accession": expr_combined.columns,
        "split": split_name.upper(),
        "species": species_name
    })
    metadata_df.to_csv(output_metadata, index=False)
    print(f"   ✅ Saved metadata: {len(metadata_df):,} samples → {output_metadata}")
    
    # Update global seen samples
    global_seen_samples.update(expr_combined.columns)
    
    return len(metadata_df), metadata_df, global_seen_samples

# ============================================================
# 4. EXTRACT ALL SPLITS FOR BOTH SPECIES WITH CROSS-SPLIT DEDUP
# ============================================================
print(f"\n🚀 Starting fast vectorized extraction with cross-split deduplication...\n")
start = time.time()

split_results = {
    "train": {"metadata": None, "count": 0},
    "val": {"metadata": None, "count": 0},
    "test": {"metadata": None, "count": 0},
}

# Global set to track samples across all splits
global_seen_samples = set()

for split_name, n_samples in [("train", TRAIN_SAMPLES), ("val", VAL_SAMPLES), ("test", TEST_SAMPLES)]:
    print(f"\n{'#'*70}")
    print(f"# SPLIT: {split_name.upper()}")
    print(f"{'#'*70}")
    
    split_dir = os.path.join(output_dir, split_name)
    
    human_expr_path = os.path.join(split_dir, f"expression_{split_name}_human.parquet")
    human_meta_path = os.path.join(split_dir, f"metadata_{split_name}_human.csv")
    
    mouse_expr_path = os.path.join(split_dir, f"expression_{split_name}_mouse.parquet")
    mouse_meta_path = os.path.join(split_dir, f"metadata_{split_name}_mouse.csv")
    
    # Extract HUMAN (with global dedup)
    human_count, human_meta, global_seen_samples = extract_and_normalize_streaming(
        human_h5_path, "HUMAN", split_name, n_samples,
        gene_lengths_human, canonical_ortho_genes_list,
        human_expr_path, human_meta_path,
        global_seen_samples,
        batch_size=EXTRACTION_BATCH_SIZE,
        random_seed=42 + hash(split_name) % 1000
    )
    
    # Extract MOUSE (with global dedup)
    mouse_count, mouse_meta, global_seen_samples = extract_and_normalize_streaming(
        mouse_h5_path, "MOUSE", split_name, n_samples,
        gene_lengths_mouse, canonical_ortho_genes_list,
        mouse_expr_path, mouse_meta_path,
        global_seen_samples,
        batch_size=EXTRACTION_BATCH_SIZE,
        random_seed=43 + hash(split_name) % 1000
    )
    
    # Combine metadata
    all_metas = []
    if human_meta is not None:
        all_metas.append(human_meta)
    if mouse_meta is not None:
        all_metas.append(mouse_meta)
    
    if all_metas:
        combined_meta = pd.concat(all_metas, ignore_index=True)
        combined_meta_path = os.path.join(split_dir, f"metadata_{split_name}.csv")
        combined_meta.to_csv(combined_meta_path, index=False)
        
        print(f"\n{'='*70}")
        print(f"✅ {split_name.upper()} COMPLETE")
        print(f"{'='*70}")
        print(f"   Total: {len(combined_meta):,}")
        print(f"   HUMAN: {len(combined_meta[combined_meta['species']=='HUMAN']):,}")
        print(f"   MOUSE: {len(combined_meta[combined_meta['species']=='MOUSE']):,}")
        print(f"   Global seen samples: {len(global_seen_samples):,}")
        
        split_results[split_name]["metadata"] = combined_meta
        split_results[split_name]["count"] = len(combined_meta)

# ============================================================
# 5. SUMMARY WITH DEDUPLICATION STATISTICS
# ============================================================
elapsed = (time.time() - start) / 60

print(f"\n{'='*70}")
print(f"✅ EXTRACTION COMPLETE (WITH CROSS-SPLIT DEDUPLICATION)")
print(f"{'='*70}\n")

total_samples = 0
for split in ["train", "val", "test"]:
    if split_results[split]["count"] > 0:
        meta = split_results[split]["metadata"]
        print(f"{split.upper()}: {split_results[split]['count']:,} samples")
        print(f"   HUMAN: {len(meta[meta['species']=='HUMAN']):,}, MOUSE: {len(meta[meta['species']=='MOUSE']):,}\n")
        total_samples += split_results[split]["count"]

print(f"📊 Total samples (all splits): {total_samples:,}")
print(f"🔒 Cross-split duplicates prevented: YES")
print(f"⏱️ Total time: {elapsed:.1f} minutes")
print(f"\n📂 Output in {output_dir}/")



SECTION 4.1: EXTRACT & NORMALIZE WITH VECTORIZED BATCHING

⚙️ Configuration:
   Train samples: 100000
   Val samples: 10000
   Test samples: 10000
   Batch size: 10000 (vectorized processing)
   QC threshold: 14,000 non-zero genes

📂 Loading exon lengths...
   Human: 77,078 genes
   Mouse: 77,969 genes

✅ Output directory: ../data/archs4/train_orthologs
🧬 Canonical genes: 16,109

🚀 Starting fast vectorized extraction with cross-split deduplication...


######################################################################
# SPLIT: TRAIN
######################################################################

🔄 HUMAN - TRAIN
Extracting 100,000 samples (vectorized batches of 10,000)

  📦 Batch 1: Requesting 10,000 samples...


100%|██████████| 10000/10000 [00:26<00:00, 372.60it/s]


     ✅ Received 10,000 samples
     ✅ Aggregated gene duplicates: 62,548 unique genes
     ✅ After QC & dedup: 8,456 new
     📊 Progress: 8,456 total new samples extracted

  📦 Batch 2: Requesting 10,000 samples...


100%|██████████| 10000/10000 [00:30<00:00, 328.49it/s]


     ✅ Received 10,000 samples
     ✅ Aggregated gene duplicates: 62,548 unique genes
     ✅ After QC & dedup: 8,264 new
        (local_dups=146, global_dups=0)
     📊 Progress: 16,720 total new samples extracted

  📦 Batch 3: Requesting 10,000 samples...


...

     ✅ Received 3 samples
     ✅ Aggregated gene duplicates: 53,454 unique genes
     ⚠️  All 1 samples already in other splits! Skipping batch.

  📦 Batch 13: Requesting 3 samples...


100%|██████████| 3/3 [00:00<00:00, 558.12it/s]


     ✅ Received 3 samples
     ✅ Aggregated gene duplicates: 53,454 unique genes
     ✅ After QC & dedup: 2 new
     📊 Progress: 9,999 total new samples extracted

  📦 Batch 14: Requesting 1 samples...


100%|██████████| 1/1 [00:00<00:00, 50.46it/s]


     ✅ Received 1 samples
     ✅ Aggregated gene duplicates: 53,454 unique genes
     ✅ After QC & dedup: 1 new
     📊 Progress: 10,000 total new samples extracted

💾 Combining 12 batches and saving...
   ✅ Saved expression: (16109, 10000) → ../data/archs4/train_orthologs/test/expression_test_mouse.parquet
   ✅ Saved metadata: 10,000 samples → ../data/archs4/train_orthologs/test/metadata_test_mouse.csv

✅ TEST COMPLETE
   Total: 20,000
   HUMAN: 10,000
   MOUSE: 10,000
   Global seen samples: 240,000

✅ EXTRACTION COMPLETE (WITH CROSS-SPLIT DEDUPLICATION)

TRAIN: 200,000 samples
   HUMAN: 100,000, MOUSE: 100,000

VAL: 20,000 samples
   HUMAN: 10,000, MOUSE: 10,000

TEST: 20,000 samples
   HUMAN: 10,000, MOUSE: 10,000

📊 Total samples (all splits): 240,000
🔒 Cross-split duplicates prevented: YES
⏱️ Total time: 65.9 minutes

📂 Output in ../data/archs4/train_orthologs/


In [ ]:
"""
SECTION 4.2: ExpressionDataset - PyTorch Dataset for Split-Based Expression Data
Loads from split parquet files (with optional species-specific files) with normalization
"""

print("\n" + "=" * 70)
print("SECTION 4.2: EXPRESSION DATASET FOR TRAINING")
print("=" * 70)

import torch
from torch.utils.data import Dataset

class ExpressionDataset(Dataset):
    """
    PyTorch dataset for expression data loaded from split parquet files.
    
    Features:
    - Loads from split-specific parquet file(s) 
    - Can load single combined file or separate human/mouse files
    - DDP compatible with DistributedSampler
    - Optional z-score normalization per sample
    
    Args:
        expr_parquet: Path to expression matrix parquet, or list of paths
        meta_csv: Path to metadata CSV
        standardize: Apply z-score normalization (True/False)
    """
    
    def __init__(self, expr_parquet, meta_csv, standardize=True):
        print(f"\n✅ Loading expression data...")
        
        # Handle both single file and list of files
        if isinstance(expr_parquet, str):
            print(f"   Loading: {os.path.basename(expr_parquet)}")
            self.expr_matrix = pd.read_parquet(expr_parquet)
        elif isinstance(expr_parquet, list):
            print(f"   Loading {len(expr_parquet)} parquet files:")
            expr_dfs = []
            for path in expr_parquet:
                print(f"      - {os.path.basename(path)}")
                expr_dfs.append(pd.read_parquet(path))
            # Concatenate along columns (samples)
            self.expr_matrix = pd.concat(expr_dfs, axis=1)
            print(f"   Combined shape: {self.expr_matrix.shape}")
        
        print(f"   Expression matrix: {self.expr_matrix.shape} (genes × samples)")
        
        # Load metadata
        self.metadata = pd.read_csv(meta_csv)
        
        # Verify consistency
        assert len(self.metadata) == self.expr_matrix.shape[1], \
            f"Sample count mismatch! Metadata: {len(self.metadata)}, Expression: {self.expr_matrix.shape[1]}"
        
        # Sample IDs (column order matches parquet)
        self.sample_ids = self.expr_matrix.columns.tolist()
        self.standardize = standardize
        
        print(f"   Samples: {len(self.sample_ids):,}")
        if "species" in self.metadata.columns:
            human_count = len(self.metadata[self.metadata["species"] == "HUMAN"])
            mouse_count = len(self.metadata[self.metadata["species"] == "MOUSE"])
            print(f"   HUMAN: {human_count:,}, MOUSE: {mouse_count:,}")
        
        if "split" in self.metadata.columns:
            split_name = self.metadata["split"].iloc[0] if len(self.metadata) > 0 else "UNKNOWN"
            print(f"   Split: {split_name}")
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        """
        Get one sample.
        
        Returns:
            tensor: [num_genes] expression values (optionally standardized)
        """
        sample_id = self.sample_ids[idx]
        
        # Get expression vector for this sample
        expr = self.expr_matrix[sample_id].values.astype(np.float32)
        
        # Standardize if requested
        if self.standardize:
            expr = (expr - expr.mean()) / (expr.std() + 1e-8)
            expr = np.nan_to_num(expr, nan=0.0)
        
        return torch.from_numpy(expr)

# ============================================================
# CREATE DATASETS FOR EACH SPLIT
# ============================================================
print("\n" + "=" * 70)
print("Creating datasets from split parquet files...")
print("=" * 70)

# Paths to split files
archs4_dir = "../data/archs4"
splits_base_dir = os.path.join(archs4_dir, "train_orthologs")

# Define splits with paths to human/mouse parquet files
splits_config = {
    "TRAIN": {
        "human": os.path.join(splits_base_dir, "train", "expression_train_human.parquet"),
        "mouse": os.path.join(splits_base_dir, "train", "expression_train_mouse.parquet"),
        "meta": os.path.join(splits_base_dir, "train", "metadata_train.csv"),
        # Legacy single-file path (for backwards compatibility)
        "combined": os.path.join(splits_base_dir, "train", "expression_train.parquet")
    },
    "VAL": {
        "human": os.path.join(splits_base_dir, "val", "expression_val_human.parquet"),
        "mouse": os.path.join(splits_base_dir, "val", "expression_val_mouse.parquet"),
        "meta": os.path.join(splits_base_dir, "val", "metadata_val.csv"),
        "combined": os.path.join(splits_base_dir, "val", "expression_val.parquet")
    },
    "TEST": {
        "human": os.path.join(splits_base_dir, "test", "expression_test_human.parquet"),
        "mouse": os.path.join(splits_base_dir, "test", "expression_test_mouse.parquet"),
        "meta": os.path.join(splits_base_dir, "test", "metadata_test.csv"),
        "combined": os.path.join(splits_base_dir, "test", "expression_test.parquet")
    }
}

# Create datasets
datasets = {}

for split_name, paths in splits_config.items():
    print(f"\n📦 Loading {split_name} dataset...")
    
    # Check which files exist
    human_exists = os.path.exists(paths["human"])
    mouse_exists = os.path.exists(paths["mouse"])
    combined_exists = os.path.exists(paths["combined"])
    meta_exists = os.path.exists(paths["meta"])
    
    if not meta_exists:
        print(f"   ⚠️  Metadata not found. Skipping.")
        continue
    
    # Determine which expression files to use
    if human_exists and mouse_exists:
        # Use separate human/mouse files
        print(f"   Loading human + mouse files...")
        expr_paths = [paths["human"], paths["mouse"]]
        datasets[split_name] = ExpressionDataset(expr_paths, paths["meta"], standardize=True)
    elif combined_exists:
        # Use combined file (backwards compatibility)
        print(f"   Loading combined file...")
        datasets[split_name] = ExpressionDataset(paths["combined"], paths["meta"], standardize=True)
    else:
        print(f"   ⚠️  No expression files found. Skipping.")
        continue

# Show summary
print(f"\n" + "="*70)
print(f"✅ Dataset Summary")
print(f"="*70)
for split_name, dataset in datasets.items():
    print(f"{split_name}: {len(dataset):,} samples")

# Test loading a sample if at least one dataset exists
if datasets:
    first_split = list(datasets.keys())[0]
    sample = datasets[first_split][0]
    print(f"\nSample verification (from {first_split}):")
    print(f"   Shape: {sample.shape} (one sample)")
    print(f"   Mean: {sample.mean():.6f}")
    print(f"   Std: {sample.std():.6f}")
    print(f"   Min: {sample.min():.6f}, Max: {sample.max():.6f}")

# Make datasets globally accessible
train_dataset = datasets.get("TRAIN")
val_dataset = datasets.get("VAL")
test_dataset = datasets.get("TEST")

if train_dataset:
    print(f"\n✅ Use 'train_dataset' for training ({len(train_dataset):,} samples)")
if val_dataset:
    print(f"✅ Use 'val_dataset' for validation ({len(val_dataset):,} samples)")
if test_dataset:
    print(f"✅ Use 'test_dataset' for testing ({len(test_dataset):,} samples)")



SECTION 4.2: EXPRESSION DATASET FOR TRAINING

Creating datasets from split parquet files...

📦 Loading TRAIN dataset...
   Loading human + mouse files...

✅ Loading expression data...
   Loading 2 parquet files:
      - expression_train_human.parquet
      - expression_train_mouse.parquet
   Combined shape: (16109, 200000)
   Expression matrix: (16109, 200000) (genes × samples)
   Samples: 200,000
   HUMAN: 100,000, MOUSE: 100,000
   Split: TRAIN

📦 Loading VAL dataset...
   Loading human + mouse files...

✅ Loading expression data...
   Loading 2 parquet files:
      - expression_val_human.parquet
      - expression_val_mouse.parquet
   Combined shape: (16109, 20000)
   Expression matrix: (16109, 20000) (genes × samples)
   Samples: 20,000
   HUMAN: 10,000, MOUSE: 10,000
   Split: VAL

📦 Loading TEST dataset...
   Loading human + mouse files...

✅ Loading expression data...
   Loading 2 parquet files:
      - expression_test_human.parquet
      - expression_test_mouse.parquet
   Comb

In [8]:
"""
SECTION 4.3: DDP-Compatible DataLoader Setup
Example config for single-GPU and multi-GPU training with split-based data
"""

print("\n" + "=" * 70)
print("SECTION 4.3: DATALOADER SETUP")
print("=" * 70)

from torch.utils.data import DataLoader, DistributedSampler

# ============================================================
# DATALOADER CONFIGURATION
# ============================================================
BATCH_SIZE = 32
NUM_WORKERS = 4

print(f"\n🔧 DataLoader configuration:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Num workers: {NUM_WORKERS}")
print(f"   Pin memory: True")

# ============================================================
# MULTI-GPU DDP DETECTION & DATALOADER SETUP
# ============================================================
print(f"\n📦 Setting up DataLoaders (DDP-compatible):")

# Check if running in DDP context
try:
    import torch.distributed as dist
    ddp_enabled = dist.is_available() and dist.is_initialized()
    rank = dist.get_rank() if ddp_enabled else 0
    world_size = dist.get_world_size() if ddp_enabled else 1
except:
    ddp_enabled = False
    rank = 0
    world_size = 1

if ddp_enabled:
    print(f"   🔗 DDP Mode: Rank {rank}/{world_size}")
else:
    print(f"   🖥️  Single-GPU Mode")

dataloaders = {}

# Create dataloaders with appropriate samplers
if train_dataset:
    if ddp_enabled:
        # DDP: Use DistributedSampler for synchronized multi-GPU training
        train_sampler = DistributedSampler(
            train_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=42,
            drop_last=True
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=train_sampler,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=True
        )
    else:
        # Single GPU: Use standard shuffle
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=False
        )
    
    dataloaders["train"] = train_loader
    print(f"   Train: {len(train_loader)} batches")

if val_dataset:
    if ddp_enabled:
        # DDP: Validation only on rank 0 or use DistributedSampler
        val_sampler = DistributedSampler(
            val_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=False,
            seed=42,
            drop_last=False
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            sampler=val_sampler,
            num_workers=NUM_WORKERS // 2,
            pin_memory=True
        )
    else:
        # Single GPU: No shuffle for validation
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS // 2,
            pin_memory=True
        )
    
    dataloaders["val"] = val_loader
    print(f"   Val: {len(val_loader)} batches")

if test_dataset:
    if ddp_enabled:
        # DDP: Test set distributed across ranks
        test_sampler = DistributedSampler(
            test_dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=False,
            seed=42,
            drop_last=False
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            sampler=test_sampler,
            pin_memory=True
        )
    else:
        # Single GPU: Standard test loader
        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )
    
    dataloaders["test"] = test_loader
    print(f"   Test: {len(test_loader)} batches")

# Test iteration on available loader
if dataloaders:
    first_loader = next(iter(dataloaders.values()))
    batch = next(iter(first_loader))
    print(f"\n✅ Batch shape: {batch.shape} (batch_size × num_genes)")

# Store samplers for epoch management during training
train_sampler = train_loader.sampler if ddp_enabled and train_dataset else None
val_sampler = val_loader.sampler if ddp_enabled and val_dataset else None
test_sampler = test_loader.sampler if ddp_enabled and test_dataset else None

# ============================================================
# MEMORY ANALYSIS
# ============================================================
num_genes = len(canonical_ortho_genes_list)
print(f"\n📊 Memory Analysis:")
print(f"   Genes per sample: {num_genes:,}")
print(f"   Single sample: {num_genes * 4 / 1e6:.2f} MB (float32)")
print(f"   Batch ({BATCH_SIZE}): {BATCH_SIZE * num_genes * 4 / 1e6:.2f} MB")

if train_dataset:
    train_size = len(train_dataset) * num_genes * 4 / 1e9
    print(f"   Training set in memory: ~{train_size:.2f} GB")
if val_dataset:
    val_size = len(val_dataset) * num_genes * 4 / 1e9
    print(f"   Validation set in memory: ~{val_size:.2f} GB")
if test_dataset:
    test_size = len(test_dataset) * num_genes * 4 / 1e9
    print(f"   Test set in memory: ~{test_size:.2f} GB")

# ============================================================
# TRAINING LOOP TEMPLATE WITH DDP SUPPORT
# ============================================================
print(f"\n🔗 Training Configuration:")
if ddp_enabled:
    print(f"   Mode: Multi-GPU DDP (rank {rank}/{world_size})")
    print(f"   Global batch size: {BATCH_SIZE * world_size}")
    print(f"   Local batch size: {BATCH_SIZE}")
else:
    print(f"   Mode: Single-GPU")
    print(f"   Batch size: {BATCH_SIZE}")

print(f"\n📝 Training loop template (use with dataloaders):")
print(f"""
# Training epoch loop
for epoch in range(NUM_EPOCHS):
    # Synchronize samplers across ranks (important for DDP!)
    if ddp_enabled and train_dataset:
        train_sampler.set_epoch(epoch)
    
    # Training phase
    if train_dataset:
        for batch_idx, batch in enumerate(dataloaders['train']):
            batch = batch.to(DEVICE)
            # Forward pass, loss, backward, optimizer step
            # ...
    
    # Validation phase (optional: only on rank 0)
    if val_dataset and (not ddp_enabled or rank == 0):
        with torch.no_grad():
            for val_batch in dataloaders['val']:
                val_batch = val_batch.to(DEVICE)
                # Validation inference
                # ...

# Cleanup DDP
if ddp_enabled:
    dist.destroy_process_group()
""")

print(f"\n✅ DataLoader setup complete! Ready for training.")
print(f"\n🚀 Launch training with:")
if ddp_enabled:
    print(f"   torchrun --nproc_per_node=2 your_training_script.py")
else:
    print(f"   python your_training_script.py")
print(f"\nAvailable dataloaders:")
for split_name in dataloaders.keys():
    print(f"   - dataloaders['{split_name}']")



SECTION 4.3: DATALOADER SETUP

🔧 DataLoader configuration:
   Batch size: 32
   Num workers: 4
   Pin memory: True

📦 Setting up DataLoaders (DDP-compatible):
   🖥️  Single-GPU Mode
   Train: 6250 batches
   Val: 625 batches
   Test: 625 batches

✅ Batch shape: torch.Size([32, 16109]) (batch_size × num_genes)

📊 Memory Analysis:
   Genes per sample: 16,109
   Single sample: 0.06 MB (float32)
   Batch (32): 2.06 MB
   Training set in memory: ~12.89 GB
   Validation set in memory: ~1.29 GB
   Test set in memory: ~1.29 GB

🔗 Training Configuration:
   Mode: Single-GPU
   Batch size: 32

📝 Training loop template (use with dataloaders):

# Training epoch loop
for epoch in range(NUM_EPOCHS):
    # Synchronize samplers across ranks (important for DDP!)
    if ddp_enabled and train_dataset:
        train_sampler.set_epoch(epoch)
    
    # Training phase
    if train_dataset:
        for batch_idx, batch in enumerate(dataloaders['train']):
            batch = batch.to(DEVICE)
            # Fo